# 🛡️ Agentic KYC Intelligence Platform
This notebook launches a Streamlit UI connected to your local vLLM/OpenAI-compatible LLM.

In [2]:
pip install -q --root-user-action=ignore streamlit requests openai vllm pyngrok

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\GenAIHYDRPUSR25\\AppData\\Local\\Temp\\pip-install-odb1t444\\vllm_0cf7b749d282420e92eed7ec5f5c923e\\vllm\\model_executor\\layers\\mamba\\ops\\configs\\selective_state_update\\headdim=64,dstate=128,device_name=NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition,cache_dtype=float16.json'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1 — Start vLLM Server

In [ ]:
# Run this in a separate terminal if needed

# Example:
# !vllm serve Qwen/Qwen2-7B-Instruct --port 8000


## Step 2 — Create Streamlit App

In [ ]:
streamlit_code = r"""
import streamlit as st
import requests

BASE_URL = "http://localhost:8000/v1"

st.set_page_config(
    page_title="KYC Intelligence Platform",
    page_icon="🛡️",
    layout="wide"
)

st.title("🛡️ Agentic KYC Intelligence Platform")
st.caption("Powered by Local vLLM Model")

def get_model():
    try:
        r = requests.get(f"{BASE_URL}/models", timeout=10)
        r.raise_for_status()
        data = r.json()
        return data["data"][0]["id"]
    except Exception as e:
        st.error(f"Could not connect to model server: {e}")
        return None

def analyze_kyc(model_name, prompt):
    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "system",
                "content": "You are an expert KYC and AML analyst."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.3,
        "max_tokens": 512
    }

    r = requests.post(
        f"{BASE_URL}/chat/completions",
        json=payload,
        timeout=120
    )

    r.raise_for_status()

    return r.json()["choices"][0]["message"]["content"]

model_name = get_model()

if model_name:
    st.sidebar.success(f"Connected Model: {model_name}")
else:
    st.stop()

st.header("Customer Information")

col1, col2 = st.columns(2)

with col1:
    name = st.text_input("Customer Name")
    age = st.number_input("Age", 18, 100, 30)
    income = st.number_input("Annual Income", 0, 100000000, 500000)

with col2:
    credit_score = st.number_input("Credit Score", 300, 900, 700)
    defaults = st.number_input("Number of Defaults", 0, 20, 0)

employment = st.selectbox(
    "Employment Status",
    ["employed", "self-employed", "student", "retired", "unemployed"]
)

bankruptcy = st.selectbox(
    "Previous Bankruptcy",
    ["No", "Yes"]
)

uploaded_file = st.file_uploader(
    "Upload Supporting Documents",
    type=["pdf", "png", "jpg", "jpeg"]
)

if st.button("Run AI KYC Analysis"):

    prompt = f"""
    Analyze this KYC applicant:

    Name: {name}
    Age: {age}
    Income: {income}
    Credit Score: {credit_score}
    Defaults: {defaults}
    Employment: {employment}
    Bankruptcy: {bankruptcy}

    Give:
    1. Risk Score
    2. AML Concerns
    3. Fraud Indicators
    4. Financial Stability
    5. Final Recommendation
    """

    with st.spinner("Analyzing..."):
        try:
            result = analyze_kyc(model_name, prompt)

            st.success("Analysis Completed")

            st.markdown(result)

        except Exception as e:
            st.error(str(e))
"""

with open("streamlit_kyc_app.py", "w") as f:
    f.write(streamlit_code)

print("streamlit_kyc_app.py created successfully")


## Step 3 — Launch Streamlit

In [ ]:
!streamlit run streamlit_kyc_app.py